# 04 · Baseline classifiers + 5-fold cross-validation (RQ1)
Logistic Regression, Decision Tree, Random Forest, Extra Trees, SVM (RBF), KNN, XGBoost with default hyperparameters.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "ml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, json
print("project root:", ROOT)

In [ ]:
from ml.models.baselines import get_baseline_models
from ml.training.evaluate import evaluate_model, cross_validate_classifier
from sklearn.base import clone
import time
s = joblib.load("ml/data/splits/dataset_splits.joblib")
X_train, y_train, X_test, y_test = s["X_train"], s["y_train"], s["X_test"], s["y_test"]
rows = []
for name, model in get_baseline_models(random_state=42).items():
    t = time.perf_counter(); model.fit(X_train, y_train); tt = time.perf_counter() - t
    ev = evaluate_model(model, X_test, y_test, name, training_time=tt)
    cvm = clone(model)
    if name == "support_vector_machine": cvm.set_params(probability=False)
    cv = cross_validate_classifier(cvm, X_train, y_train, name, n_splits=5)
    rows.append({"model": name, "test_f1": ev["f1"], "cv_f1": cv["f1_mean"], "cv_std": cv["f1_std"], "recall": ev["recall"], "fpr": ev["false_positive_rate"], "train_s": round(tt, 1), "infer_ms_1k": ev["inference_time_ms_per_1k"]})
    print(f"{name:26s} test F1 {ev['f1']:.4f} | CV {cv['f1_mean']:.4f} ± {cv['f1_std']:.4f}")
pd.DataFrame(rows).sort_values("test_f1", ascending=False)